# S1-S5 breast cancer contour application

This advanced `pyXenium.contour` tutorial uses the Atera Xenium WTA FFPE breast cancer export at
`Y:\long\10X_datasets\Xenium\Atera\WTA_Preview_FFPE_Breast_Cancer_outs` to build five HistoSeg-backed contour annotations and then run three contour-native biology analyses.

The five annotation classes are intentionally broader than the basic contour tutorial:

| S class | Biological compartment | Aggregated Atera cell groups |
|---|---|---|
| S1 | Invasive tumor plus invasion-associated CAFs | CAFs Invasive Associated, 11q13 invasive tumor, G1/S, mitotic |
| S2 | Basal-like structured DCIS plus lymphoid/DC | Basal DCIS, dendritic cells, B cells, T lymphocytes |
| S3 | Myeloid-vascular-stromal niche | Mast/myeloid/macrophage, CXCL14+ fibroblast, endothelial, pericyte |
| S4 | Myoepithelial/DCIS CAF/plasma compartment | Myoepithelial, DCIS-associated CAFs, plasma cells |
| S5 | Apocrine/luminal amorphous DCIS axis | Apocrine and luminal-like amorphous DCIS |

The RTD snapshot below uses a curated candidate transcript panel to keep the committed notebook small. The same code path can be switched to a full unfiltered transcript stream for local genome-wide scans.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import sys
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.stats import kruskal


def _find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'src' / 'pyXenium').exists():
            return candidate
    return cwd

PROJECT_ROOT = _find_repo_root()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from pyXenium.contour import (
    add_contours_from_geojson,
    compare_contour_cell_composition,
    compare_contour_transcript_de,
    generate_barrier_contour_shells,
    generate_xenium_explorer_annotations,
)
from pyXenium.io import XeniumFrameChunkSource, read_xenium
from pyXenium.io.api import TRANSCRIPT_POINT_COLUMNS, TRANSCRIPT_POINT_COLUMN_TYPES
from pyXenium.io.xenium_artifacts import iter_transcript_chunks

sns.set_theme(context='notebook', style='whitegrid')

DATASET_ROOT = Path(r'Y:\long\10X_datasets\Xenium\Atera\WTA_Preview_FFPE_Breast_Cancer_outs')
HISTOSEG_ROOT = Path(r'D:\GitHub\HistoSeg')
OUTPUT_ROOT = DATASET_ROOT / 'pyxenium_tutorial_outputs' / 'atera_s1_s5_contour_application'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
STATIC_SNAPSHOT_DIR = PROJECT_ROOT / 'docs' / '_static' / 'tutorials' / 'contour_s1_s5'
GRAPHCLUST_RELPATH = Path('analysis/analysis/clustering/gene_expression_graphclust/clusters.csv')
CELL_GROUPS_PATH = DATASET_ROOT / 'WTA_Preview_FFPE_Breast_Cancer_cell_groups.csv'
CELLS_PARQUET_RELPATH = Path('cells.parquet')
TRANSCRIPTS_PATH = DATASET_ROOT / 'transcripts.parquet'
S1_S5_GEOJSON = DATASET_ROOT / 'xenium_explorer_annotations.s1_s5.generated.geojson'
PIXEL_SIZE_UM = 0.2125

RUN_HISTOSEG_EXPORT = False
RUN_FULL_TRANSCRIPT_DE = False
RUN_TRANSCRIPT_SNAPSHOT = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
STATIC_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)


def _safe_fdr(p_values: pd.Series) -> pd.Series:
    values = p_values.to_numpy(dtype=float)
    adjusted = np.full(len(values), np.nan, dtype=float)
    finite_mask = np.isfinite(values)
    finite = values[finite_mask]
    if finite.size:
        order = np.argsort(finite)
        ranked = finite[order] * finite.size / np.arange(1, finite.size + 1, dtype=float)
        ranked = np.minimum.accumulate(ranked[::-1])[::-1]
        local = np.empty_like(finite)
        local[order] = np.minimum(ranked, 1.0)
        adjusted[finite_mask] = local
    return pd.Series(adjusted, index=p_values.index)


def boundary_global_stats(curve_source: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for gene, local in curve_source.groupby('gene', sort=True):
        arrays = []
        means = {}
        for group in ['S1', 'S4', 'S5']:
            values = local.loc[local['assigned_structure'].eq(group), 'density_per_um2'].to_numpy(dtype=float)
            values = values[np.isfinite(values)]
            if values.size:
                means[group] = float(np.mean(values))
            if values.size >= 2:
                arrays.append(values)
        status = 'ok'
        statistic = np.nan
        p_value = np.nan
        if len(arrays) < 2:
            status = 'insufficient_contour_replicates'
        else:
            try:
                statistic, p_value = kruskal(*arrays, nan_policy='omit')
            except ValueError:
                status = 'test_not_defined'
        top = max(means, key=means.get) if means else None
        bottom = min(means, key=means.get) if means else None
        rows.append(
            {
                'gene': gene,
                'comparison': 'S1_vs_S4_vs_S5_outward_0_60um',
                'top_group': top,
                'bottom_group': bottom,
                'max_mean_density_per_um2': np.nan if top is None else means[top],
                'min_mean_density_per_um2': np.nan if bottom is None else means[bottom],
                'delta_density_per_um2': np.nan if top is None or bottom is None else means[top] - means[bottom],
                'statistic': statistic,
                'p_value': p_value,
                'status': status,
            }
        )
    frame = pd.DataFrame(rows)
    frame['fdr'] = _safe_fdr(frame['p_value'])
    return frame.sort_values(['fdr', 'p_value', 'delta_density_per_um2'], ascending=[True, True, False], na_position='last')



## 1. Build the S1-S5 structure definitions

The official GraphClust table stores numeric cluster IDs, while the Atera cell-group CSV stores readable labels and colors. We bridge those tables and aggregate cluster IDs into the five S classes. Two user-facing names are aliased to the exact dataset labels: `CAFs Invasive Associated` -> `CAFs, Invasive Associated` and `CAFs DCIS Associated` -> `CAFs, DCIS Associated`.


In [ ]:
S1_S5_PANEL = {
    'S1': [
        'CAFs Invasive Associated',
        '11q13 Invasive Tumor Cells',
        '11q13 Invasive Tumor Cells (G1/S)',
        '11q13 Invasive Tumor Cells (Mitotic)',
    ],
    'S2': [
        'Basal-like Structured DCIS Cells',
        'Dendritic Cells',
        'B Cells',
        'T Lymphocytes',
    ],
    'S3': [
        'Mast Cells',
        'Myeloid Cells',
        'Macrophages',
        'CXCL14+ Fibroblasts',
        'Endothelial Cells',
        'Pericytes',
    ],
    'S4': [
        'Myoepithelial Cells',
        'CAFs DCIS Associated',
        'Plasma Cells',
    ],
    'S5': [
        'Apocrine Cells',
        'Luminal-like Amorphous DCIS Cells',
    ],
}

LABEL_ALIASES = {
    'CAFs Invasive Associated': 'CAFs, Invasive Associated',
    'CAFs DCIS Associated': 'CAFs, DCIS Associated',
}
S_COLORS = {'S1': '#C43C39', 'S2': '#3F7CAC', 'S3': '#4A9D67', 'S4': '#B9802D', 'S5': '#9A5BC7'}
S_DESCRIPTIONS = {
    'S1': 'Invasive tumor + CAFs',
    'S2': 'Basal DCIS + lymphoid',
    'S3': 'Myeloid-vascular-stromal',
    'S4': 'Myoepithelial/DCIS CAF/plasma',
    'S5': 'Apocrine/luminal DCIS',
}

cluster_df = pd.read_csv(DATASET_ROOT / GRAPHCLUST_RELPATH)
cell_groups_df = pd.read_csv(CELL_GROUPS_PATH).rename(columns={'cell_id': 'Barcode', 'group': 'cell_group'})
cluster_bridge = cluster_df.merge(
    cell_groups_df[['Barcode', 'cell_group', 'color']],
    on='Barcode',
    how='inner',
)
label_to_clusters = (
    cluster_bridge.groupby('cell_group')['Cluster']
    .apply(lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist()))
    .to_dict()
)
label_counts = cluster_bridge.groupby('cell_group').size().to_dict()

structures = []
structure_rows = []
for idx, (structure_name, requested_labels) in enumerate(S1_S5_PANEL.items(), start=1):
    cluster_ids = []
    resolved_labels = []
    input_cell_count = 0
    for requested_label in requested_labels:
        resolved_label = LABEL_ALIASES.get(requested_label, requested_label)
        if resolved_label not in label_to_clusters:
            raise KeyError(f'Missing Atera cell group: {requested_label!r} -> {resolved_label!r}')
        cluster_ids.extend(label_to_clusters[resolved_label])
        resolved_labels.append(resolved_label)
        input_cell_count += int(label_counts[resolved_label])

    structures.append(
        {
            'structure_name': structure_name,
            'cluster_ids': sorted(set(cluster_ids)),
            'structure_color': S_COLORS[structure_name],
            'structure_id': idx,
            'member_labels': tuple(resolved_labels),
        }
    )
    structure_rows.append(
        {
            'structure_name': structure_name,
            'description': S_DESCRIPTIONS[structure_name],
            'cluster_ids': sorted(set(cluster_ids)),
            'member_labels': ', '.join(resolved_labels),
            'input_cell_count': input_cell_count,
        }
    )

structure_definition_table = pd.DataFrame(structure_rows)
display(structure_definition_table)


## 2. Generate HistoSeg-backed Xenium Explorer annotations

`generate_xenium_explorer_annotations(...)` calls the HistoSeg multi-structure contour API and writes a Xenium Explorer-compatible GeoJSON. The legacy `xenium_explorer_annotations.geojson` is left untouched; this tutorial writes `xenium_explorer_annotations.s1_s5.generated.geojson`.


In [ ]:
if RUN_HISTOSEG_EXPORT or not S1_S5_GEOJSON.exists():
    artifacts = generate_xenium_explorer_annotations(
        DATASET_ROOT,
        structures=structures,
        output_relpath=OUTPUT_ROOT.relative_to(DATASET_ROOT),
        clusters_relpath=GRAPHCLUST_RELPATH,
        cells_parquet_relpath=CELLS_PARQUET_RELPATH,
        histoseg_root=HISTOSEG_ROOT if HISTOSEG_ROOT.exists() else None,
        min_cells=500,
        min_component_pixels=180,
        xenium_pixel_size_um=PIXEL_SIZE_UM,
    )
    shutil.copy2(artifacts['geojson'], S1_S5_GEOJSON)
else:
    artifacts = {
        'geojson': str(S1_S5_GEOJSON),
        'out_dir': str(OUTPUT_ROOT),
        'preview_png': str(OUTPUT_ROOT / 'multi_structure_contour_preview.png'),
    }

contour_counts = pd.read_csv(OUTPUT_ROOT / 's1_s5_contour_counts.csv') if (OUTPUT_ROOT / 's1_s5_contour_counts.csv').exists() else pd.read_csv(OUTPUT_ROOT / 'structure_contour_cell_counts.csv')
display(contour_counts)
artifacts


### Snapshot contour counts

The committed RTD snapshot includes this small result table, so the page remains informative even with notebook execution disabled.

| structure_name | description                   | input_cell_count | assigned_cell_count | contour_count |
| -------------- | ----------------------------- | ---------------- | ------------------- | ------------- |
| S1             | Invasive tumor + CAFs         | 72873            | 71725               | 106           |
| S2             | Basal DCIS + lymphoid         | 23298            | 15935               | 358           |
| S3             | Myeloid-vascular-stromal      | 26690            | 37038               | 491           |
| S4             | Myoepithelial/DCIS CAF/plasma | 33753            | 36486               | 313           |
| S5             | Apocrine/luminal DCIS         | 13431            | 8861                | 28            |


### Annotation preview

The generated S1-S5 export contains multiple contours per class. In the snapshot run, the HistoSeg count table reported 106 S1, 358 S2, 491 S3, 313 S4, and 28 S5 contour components before GeoJSON polygon serialization.

![S1-S5 annotation preview](../_static/tutorials/contour_s1_s5/s1_s5_annotation_preview.png)


## 3. Load Xenium, import contours, and attach cell-group labels

We load the cell table without materializing the full transcript parquet. For transcript-level examples, the notebook attaches a filtered streaming point source over the candidate gene panel. For a full local run, set `RUN_FULL_TRANSCRIPT_DE=True` to keep the original unfiltered transcript stream and use full-contour CPM denominators.


In [ ]:
sdata = read_xenium(
    str(DATASET_ROOT),
    as_='sdata',
    include_transcripts=False,
    include_boundaries=False,
    include_images=False,
    clusters_relpath=GRAPHCLUST_RELPATH,
)
cell_group_map = cell_groups_df.set_index('Barcode')['cell_group']
sdata.table.obs['cell_group'] = sdata.table.obs.index.map(cell_group_map).fillna('Unassigned')

add_contours_from_geojson(
    sdata,
    S1_S5_GEOJSON,
    key='s1_s5_contours',
    id_key='name',
    pixel_size_um=PIXEL_SIZE_UM,
)

print(sdata.component_summary())
print(sorted(sdata.shapes['s1_s5_contours']['assigned_structure'].dropna().unique()))


In [ ]:
DEG_PANEL = [
    'MMP11', 'COL11A1', 'POSTN', 'COL1A1', 'COL1A2', 'EPCAM', 'MKI67',
    'SOSTDC1', 'KRT23', 'CD3D', 'MS4A1', 'C1QA', 'CD163', 'CSF1R', 'C3',
    'CDH5', 'MMRN2', 'KDR', 'IGHA1', 'JCHAIN', 'TAT', 'HSPB8', 'CLIC6',
    'PIP', 'ESR1', 'PGR',
]


def attach_filtered_transcript_source(sdata, genes: Iterable[str]) -> None:
    gene_set = set(str(gene) for gene in genes)
    sdata.points.pop('transcripts', None)
    sdata.point_sources['transcripts'] = XeniumFrameChunkSource(
        columns=TRANSCRIPT_POINT_COLUMNS,
        column_types=TRANSCRIPT_POINT_COLUMN_TYPES,
        chunk_iter_factory=lambda: iter_transcript_chunks(str(TRANSCRIPTS_PATH), genes=gene_set),
        attrs={'source_path': str(TRANSCRIPTS_PATH), 'gene_filter': tuple(sorted(gene_set))},
        preserve_extra_columns=True,
    )

if RUN_FULL_TRANSCRIPT_DE:
    sdata_full = read_xenium(
        str(DATASET_ROOT),
        as_='sdata',
        prefer='auto',
        stream_transcripts=True,
        include_boundaries=False,
        include_images=False,
        clusters_relpath=GRAPHCLUST_RELPATH,
    )
    sdata.point_sources['transcripts'] = sdata_full.point_sources['transcripts']
else:
    attach_filtered_transcript_source(sdata, DEG_PANEL)


## 4. Transcript-level contour DEG

`compare_contour_transcript_de(...)` assigns transcript coordinates directly to contour polygons, aggregates raw contour x gene counts, computes CPM/log1p CPM at contour level, and then treats each contour as a replicate. The snapshot uses the candidate panel above; the same function can run genome-wide if the transcript stream is unfiltered.


In [ ]:
if RUN_TRANSCRIPT_SNAPSHOT:
    deg_result = compare_contour_transcript_de(
        sdata,
        contour_key='s1_s5_contours',
        groupby='assigned_structure',
        genes=DEG_PANEL,
        comparisons=('global', 'one_vs_rest'),
        transcript_query='quality_score >= 20 and valid == True',
        min_total_transcripts_per_contour=1,
        min_contours_per_group=2,
        include_zero_counts=True,
    )
    deg_result['global_de'].to_csv(OUTPUT_ROOT / 's1_s5_transcript_de_global.csv', index=False)
    deg_result['one_vs_rest_de'].to_csv(OUTPUT_ROOT / 's1_s5_transcript_de_one_vs_rest.csv', index=False)
else:
    deg_result = {
        'global_de': pd.read_csv(STATIC_SNAPSHOT_DIR / 's1_s5_transcript_de_global.csv'),
        'one_vs_rest_de': pd.read_csv(STATIC_SNAPSHOT_DIR / 's1_s5_transcript_de_one_vs_rest.csv'),
    }

display(deg_result['global_de'].head(12))
display(deg_result['one_vs_rest_de'].head(12))


### Snapshot DEG tables

Top global candidate genes by contour-level Kruskal-Wallis test:

| gene  | top_group | bottom_group | delta_log1p_cpm | fdr      |
| ----- | --------- | ------------ | --------------- | -------- |
| CDH5  | S3        | S2           | 5.17            | 1.50e-67 |
| KDR   | S3        | S2           | 4.88            | 1.35e-58 |
| MMRN2 | S3        | S2           | 4.66            | 1.71e-56 |
| CLIC6 | S5        | S2           | 8.37            | 5.14e-31 |
| PIP   | S5        | S4           | 7.85            | 1.53e-30 |
| ESR1  | S5        | S4           | 6.08            | 3.87e-30 |
| HSPB8 | S5        | S2           | 8.42            | 9.51e-30 |
| EPCAM | S5        | S4           | 4.70            | 4.11e-28 |

Top two one-vs-rest candidate genes per S class:

| case | gene  | log2fc_cpm | fdr      |
| ---- | ----- | ---------- | -------- |
| S1   | EPCAM | 1.91       | 5.25e-10 |
| S1   | TAT   | -0.96      | 5.43e-10 |
| S2   | CDH5  | -1.45      | 4.55e-32 |
| S2   | MMRN2 | -1.38      | 6.22e-28 |
| S3   | CDH5  | 2.23       | 1.62e-53 |
| S3   | KDR   | 2.12       | 3.54e-45 |
| S4   | CDH5  | -1.56      | 1.11e-13 |
| S4   | MMRN2 | -0.97      | 7.68e-12 |
| S5   | HSPB8 | 3.30       | 1.18e-11 |
| S5   | CLIC6 | 3.75       | 1.13e-10 |


### DEG visualization

The heatmap shows mean contour-level `log1p(CPM)` for the top global candidate genes. The dotplot summarizes the strongest one-vs-rest signals per S class.

![S1-S5 DEG heatmap](../_static/tutorials/contour_s1_s5/s1_s5_de_heatmap.png)

![S1-S5 DEG dotplot](../_static/tutorials/contour_s1_s5/s1_s5_de_dotplot.png)


## 5. Cell-type composition across contour classes

`compare_contour_cell_composition(...)` assigns cell centroids to every contour independently. It reports both proportions and quantities, then tests S-class differences with Kruskal-Wallis globally and Mann-Whitney tests pairwise.


In [ ]:
if RUN_TRANSCRIPT_SNAPSHOT:
    cell_result = compare_contour_cell_composition(
        sdata,
        contour_key='s1_s5_contours',
        groupby='assigned_structure',
        cell_type_key='cell_group',
    )
    cell_result['group_summary'].to_csv(OUTPUT_ROOT / 's1_s5_cell_composition_group_summary.csv', index=False)
    pd.concat(
        [
            cell_result['global_stats'].assign(test_scope='global'),
            cell_result['pairwise_stats'].assign(test_scope='pairwise'),
        ],
        ignore_index=True,
        sort=False,
    ).to_csv(OUTPUT_ROOT / 's1_s5_cell_composition_stats.csv', index=False)
else:
    cell_result = {
        'group_summary': pd.read_csv(STATIC_SNAPSHOT_DIR / 's1_s5_cell_composition_group_summary.csv'),
        'stats': pd.read_csv(STATIC_SNAPSHOT_DIR / 's1_s5_cell_composition_stats.csv'),
    }

display(cell_result['group_summary'].head(12))
if 'stats' in cell_result:
    display(cell_result['stats'].head(12))


### Snapshot cell-composition statistics

Top global cell-type composition differences across S1-S5 contours:

| cell_type                         | metric   | top_group | bottom_group | fdr      |
| --------------------------------- | -------- | --------- | ------------ | -------- |
| Pericytes                         | n_cells  | S3        | S2           | 6.01e-98 |
| Endothelial Cells                 | n_cells  | S3        | S2           | 1.40e-94 |
| Pericytes                         | fraction | S3        | S5           | 3.49e-90 |
| Endothelial Cells                 | fraction | S3        | S1           | 1.06e-89 |
| Luminal-like Amorphous DCIS Cells | n_cells  | S5        | S1           | 5.84e-77 |
| CXCL14+ Fibroblasts               | n_cells  | S1        | S2           | 5.35e-76 |
| CAFs, DCIS Associated             | fraction | S4        | S1           | 7.76e-70 |
| Luminal-like Amorphous DCIS Cells | fraction | S5        | S1           | 2.62e-67 |


### Composition visualization

![S1-S5 cell-type proportion](../_static/tutorials/contour_s1_s5/s1_s5_cell_type_proportion_barplot.png)

![S1-S5 cell-type quantity](../_static/tutorials/contour_s1_s5/s1_s5_cell_type_quantity_barplot.png)


## 6. Barrier-aware S1/S4/S5 boundary shells

The boundary question asks how S1, S4, and S5 contours behave when expanded inward and outward in 20 um steps. We generate 10 signed rings: `[-100,-80] ... [-20,0]` and `[0,20] ... [80,100]`. Outward rings are clipped by all other S1-S5 contour interiors, so a shell stops contributing when it reaches another annotated contour.


In [ ]:
selected_frame = sdata.shapes['s1_s5_contours'].loc[
    sdata.shapes['s1_s5_contours']['assigned_structure'].isin(['S1', 'S4', 'S5'])
].copy()
sdata.shapes['s1_s4_s5_contours'] = selected_frame
sdata.metadata.setdefault('contours', {})['s1_s4_s5_contours'] = {
    'units': 'micron',
    'derived_from_key': 's1_s5_contours',
}

if 's1_s4_s5_barrier_shells' not in sdata.shapes:
    generate_barrier_contour_shells(
        sdata,
        contour_key='s1_s4_s5_contours',
        barrier_contour_key='s1_s5_contours',
        inward=100.0,
        outward=100.0,
        step_size=20.0,
        output_key='s1_s4_s5_barrier_shells',
    )

shell_frame = sdata.shapes['s1_s4_s5_barrier_shells']
display(shell_frame[['contour_id', 'source_contour_id', 'assigned_structure', 'shell_start', 'shell_end', 'shell_direction']].head())
print('shell contours:', shell_frame['contour_id'].nunique())


In [ ]:
if RUN_TRANSCRIPT_SNAPSHOT:
    shell_result = compare_contour_transcript_de(
        sdata,
        contour_key='s1_s4_s5_barrier_shells',
        groupby='assigned_structure',
        genes=DEG_PANEL,
        comparisons=('global',),
        transcript_query='quality_score >= 20 and valid == True',
        min_total_transcripts_per_contour=0,
        min_contours_per_group=2,
        include_zero_counts=True,
    )
    shell_gene = shell_result['contour_gene'].copy()
    outward_0_60 = shell_gene.loc[
        shell_gene['shell_direction'].eq('outward')
        & (shell_gene['shell_start'] >= 0)
        & (shell_gene['shell_end'] <= 60)
    ].copy()
    boundary_stats = boundary_global_stats(outward_0_60)
    top_boundary_genes = boundary_stats.loc[boundary_stats['status'].eq('ok')].head(10)['gene'].tolist()
    if len(top_boundary_genes) < 10:
        top_boundary_genes = boundary_stats.head(10)['gene'].tolist()
    boundary_curves = (
        shell_gene.loc[shell_gene['gene'].isin(top_boundary_genes)]
        .groupby(['assigned_structure', 'gene', 'shell_start', 'shell_end', 'shell_mid'], as_index=False)
        .agg(
            mean_density_per_um2=('density_per_um2', 'mean'),
            sem_density_per_um2=('density_per_um2', lambda values: float(pd.Series(values).sem())),
            n_shells=('contour_id', 'nunique'),
            total_count=('count', 'sum'),
        )
        .sort_values(['gene', 'assigned_structure', 'shell_mid'], kind='stable')
    )
    boundary_stats.to_csv(OUTPUT_ROOT / 's1_s4_s5_boundary_de_global_outward_0_60.csv', index=False)
    boundary_curves.to_csv(OUTPUT_ROOT / 's1_s4_s5_barrier_ring_density_top10.csv', index=False)
else:
    boundary_curves = pd.read_csv(STATIC_SNAPSHOT_DIR / 's1_s4_s5_barrier_ring_density_top10.csv')
    boundary_stats = pd.read_csv(STATIC_SNAPSHOT_DIR / 's1_s4_s5_boundary_de_global_outward_0_60.csv')

display(boundary_stats.head(10))
display(boundary_curves.head(10))


### Snapshot boundary DE table

Top boundary genes were selected from outward 0-60 um barrier-clipped shells.

| gene   | top_group | bottom_group | delta_density_per_um2 | fdr      |
| ------ | --------- | ------------ | --------------------- | -------- |
| ESR1   | S5        | S4           | 0.004866              | 9.45e-23 |
| EPCAM  | S1        | S4           | 0.001347              | 1.24e-19 |
| HSPB8  | S5        | S1           | 0.004807              | 8.75e-15 |
| POSTN  | S1        | S5           | 0.01001               | 2.70e-10 |
| KDR    | S5        | S4           | 0.001039              | 2.70e-10 |
| COL1A1 | S5        | S4           | 0.001003              | 6.30e-09 |
| MMP11  | S1        | S5           | 0.0007454             | 4.38e-07 |
| PGR    | S1        | S5           | 0.0001369             | 5.56e-07 |
| CDH5   | S5        | S1           | 0.00189               | 5.75e-07 |
| CLIC6  | S5        | S1           | 0.0009019             | 1.32e-05 |


### Boundary density curves

Top boundary genes were selected from the outward 0-60 um interval, then plotted across the full 10 signed-distance steps. Negative distances are inside the source contour; positive distances are outside but barrier-clipped so they do not enter another contour.

![S1/S4/S5 barrier-aware density curves](../_static/tutorials/contour_s1_s5/s1_s4_s5_top10_boundary_density_curves.png)


## 7. Biological interpretation

The S1-S5 contours recover the expected spatial biology of this Atera breast cancer section.

- S1 is dominated by invasive 11q13 tumor cells and invasion-associated CAFs. In the candidate transcript scan, epithelial/tumor genes such as `EPCAM` are strongest in S1 contours, while boundary curves for `POSTN`, `COL1A1`, and `MMP11` support an invasive stromal rim around tumor-rich contours.
- S2 captures the basal-like structured DCIS and lymphoid/DC compartment. Its cell-composition profile is mixed by design, with basal DCIS plus T/B/dendritic populations, making it a useful contrast against the more vascular/myeloid S3 region.
- S3 is the clearest myeloid-vascular-stromal niche. `CDH5`, `KDR`, and `MMRN2` rank at the top of the contour DEG table, while `C1QA`, `CD163`, `CSF1R`, and `C3` support macrophage/myeloid enrichment.
- S4 combines myoepithelial cells, DCIS-associated CAFs, and plasma cells. Its composition separates it from S1 invasive CAFs and S5 luminal tumor, and it is a useful boundary comparator because it sits between epithelial and stromal biology.
- S5 is the apocrine/luminal amorphous DCIS axis. `CLIC6`, `PIP`, `ESR1`, `PGR`, and `HSPB8` form the dominant luminal/apocrine signal and remain high in the inward portions of S5 boundary shells.

A practical analysis pattern emerges: use `compare_contour_transcript_de` to identify contour-level programs, `compare_contour_cell_composition` to explain which cell types make those programs plausible, and `generate_barrier_contour_shells` plus transcript density curves to test whether a program sits inside the contour, peaks at the boundary, or extends into the surrounding microenvironment.


## 8. Output files

The lightweight RTD snapshot uses these committed artifacts:

- [`s1_s5_contour_counts.csv`](../_static/tutorials/contour_s1_s5/s1_s5_contour_counts.csv)
- [`s1_s5_transcript_de_global.csv`](../_static/tutorials/contour_s1_s5/s1_s5_transcript_de_global.csv)
- [`s1_s5_transcript_de_one_vs_rest.csv`](../_static/tutorials/contour_s1_s5/s1_s5_transcript_de_one_vs_rest.csv)
- [`s1_s5_cell_composition_stats.csv`](../_static/tutorials/contour_s1_s5/s1_s5_cell_composition_stats.csv)
- [`s1_s4_s5_barrier_ring_density_top10.csv`](../_static/tutorials/contour_s1_s5/s1_s4_s5_barrier_ring_density_top10.csv)

Local reruns also write the full working set under `pyxenium_tutorial_outputs/atera_s1_s5_contour_application/` beside the Xenium export.
